<a href="https://colab.research.google.com/github/JamesEckhartJr/Quantum-Computing-Projects/blob/main/Fixed_2_qubit_H%E2%82%82_VQE_for_Quokka.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# =============================================================================
# Fixed 2-qubit H₂ VQE for Quokka (with result explanations)
# =============================================================================

import requests
import numpy as np
from scipy.optimize import minimize

# Configuration
DEVICE = 'theq-ae31b1'                     # change if needed
URL = f'https://{DEVICE}.quokkacomputing.com/qsim/qasm'
SHOTS = 8192

# 2-qubit H₂ electronic Hamiltonian (parity mapping, STO-3G basis, ~0.735 Å)
# These coefficients do NOT include nuclear repulsion.
# Exact electronic ground-state energy ≈ -1.857 Ha
HAMILTONIAN = [
    (-1.052373, "II"),
    ( 0.397937, "IZ"),
    (-0.397937, "ZI"),
    (-0.011280, "ZZ"),
    ( 0.180931, "XX"),
]

# Approximate nuclear repulsion energy at equilibrium bond length
NUCLEAR_REPULSION = 0.715   # Ha (adds to electronic energy to give total energy)

def run_qasm(qasm: str, shots: int = SHOTS):
    payload = {"script": qasm.strip(), "count": shots}
    r = requests.post(URL, json=payload, timeout=60)
    r.raise_for_status()
    data = r.json()
    if data.get("error_code", 1) != 0:
        print("Quokka error:", data)
        return []
    return data["result"]["c"]          # list of lists: [[c0, c1], ...]

def generate_circuit(params, pauli: str) -> str:
    qasm = """OPENQASM 2.0;
include "qelib1.inc";
qreg q[2];
creg c[2];
"""
    # Simple hardware-efficient ansatz (4 parameters)
    qasm += f"ry({params[0]}) q[0];\n"
    qasm += f"ry({params[1]}) q[1];\n"
    qasm += "cx q[0], q[1];\n"
    qasm += f"ry({params[2]}) q[0];\n"
    qasm += f"ry({params[3]}) q[1];\n"

    # Basis-change rotations so we measure the desired Pauli
    for i, p in enumerate(pauli):
        if p == "X":
            qasm += f"h q[{i}];\n"
        elif p == "Y":
            qasm += f"sdg q[{i}];\n"
            qasm += f"h q[{i}];\n"

    qasm += "measure q[0] -> c[0];\n"
    qasm += "measure q[1] -> c[1];\n"
    return qasm

def expectation(pauli: str, shots_list) -> float:
    if not shots_list:
        return 0.0
    total = len(shots_list)
    exp = 0.0
    for bits in shots_list:               # bits[0] = qubit 0, bits[1] = qubit 1
        sign = 1
        for i, p in enumerate(pauli):
            if p != "I" and bits[i] == 1:
                sign *= -1
        exp += sign
    return exp / total

def compute_energy(params):
    energy = 0.0
    for coeff, pauli in HAMILTONIAN:
        if pauli == "II":
            energy += coeff               # identity term is added classically
            continue
        shots = run_qasm(generate_circuit(params, pauli))
        energy += coeff * expectation(pauli, shots)
    return energy

# ========================= RUN OPTIMIZATION =========================
print("Starting fixed 2-qubit H₂ VQE on Quokka...\n")

np.random.seed(42)
x0 = np.random.uniform(-np.pi, np.pi, 4)

result = minimize(compute_energy, x0, method="COBYLA",
                  options={"maxiter": 60, "disp": True, "rhobeg": 0.5})

# ========================= RESULTS WITH EXPLANATIONS =========================
electronic_energy = result.fun
total_energy = electronic_energy + NUCLEAR_REPULSION
exact_electronic = -1.857
exact_total = -1.137

print("\n" + "="*60)
print("VQE RESULTS FOR H₂ (2-qubit, STO-3G)")
print("="*60)
print(f"Optimized electronic energy : {electronic_energy:.6f} Ha")
print(f"Exact electronic energy     : {exact_electronic:.3f} Ha")
print(f"Electronic energy error     : {abs(electronic_energy - exact_electronic):.6f} Ha")
print()
print(f"Nuclear repulsion (approx.) : +{NUCLEAR_REPULSION:.3f} Ha")
print(f"Estimated total energy      : {total_energy:.6f} Ha")
print(f"Exact total energy (FCI)    : {exact_total:.3f} Ha")
print(f"Total energy error          : {abs(total_energy - exact_total):.6f} Ha")
print()
print(f"Optimizer evaluations       : {result.nfev}")
print(f"Optimization success        : {result.success}")
print("="*60)
print("""
EXPLANATION:
- The number the VQE directly optimizes is the *electronic* energy.
  This is the expectation value of the electronic Hamiltonian
  (kinetic energy + electron-nuclear attraction + electron-electron repulsion).

- The famous textbook value of ≈ -1.137 Ha is the *total* molecular energy.
  Total energy = electronic energy + nuclear repulsion energy.
  At the equilibrium bond length the nuclear repulsion is about +0.715 Ha,
  which is why -1.857 + 0.715 ≈ -1.142 (close to -1.137).

- A result within a few milliHartree of the exact electronic energy
  on a noiseless emulator with this shallow ansatz and shot count
  is a successful VQE demonstration.
""")

Starting fixed 2-qubit H₂ VQE on Quokka...

Return from COBYLA because the trust region radius reaches its lower bound.
Number of function values = 43   Least value of F = -1.8317375791015627
The corresponding X is:
[-0.25120879  2.22271463 -0.0657775   0.51747964]


VQE RESULTS FOR H₂ (2-qubit, STO-3G)
Optimized electronic energy : -1.831738 Ha
Exact electronic energy     : -1.857 Ha
Electronic energy error     : 0.025262 Ha

Nuclear repulsion (approx.) : +0.715 Ha
Estimated total energy      : -1.116738 Ha
Exact total energy (FCI)    : -1.137 Ha
Total energy error          : 0.020262 Ha

Optimizer evaluations       : 43
Optimization success        : True

EXPLANATION:
- The number the VQE directly optimizes is the *electronic* energy.
  This is the expectation value of the electronic Hamiltonian
  (kinetic energy + electron-nuclear attraction + electron-electron repulsion).

- The famous textbook value of ≈ -1.137 Ha is the *total* molecular energy.
  Total energy = electronic energy